In [2]:
from unstructured_client import UnstructuredClient
from unstructured_client.models import (
    operations,
    shared,
)

client = UnstructuredClient(
    api_key_auth="sA2fqF1bKetthL0H5B9zbERuc6Ffon"
)

with open("/workspaces/nlp-paper-rag/data/papers/transformers/attention_is_all_you_need.pdf", "rb") as f:
    files = shared.Files(
        content = f.read(),
        file_name = "attention_is_all_you_need.pdf"
    )

req = operations.PartitionRequest(
    partition_parameters=shared.PartitionParameters(
        files=files,
        strategy="hi_res",
    )
)

result = client.general.partition(request=req)

elements = result.elements

for e in elements[:5]:
    print(e["type"])
    print(e["text"])
    print("-" * 50)

INFO: split_pdf event=plan_created operation_id=d9d14056-b391-4555-9a43-ff331a5e56b5 filename=attention_is_all_you_need.pdf strategy=hi_res page_range=1-11 page_count=11 split_size=3 chunk_count=4 concurrency=5 allow_failed=False cache_mode=disabled timeout_seconds=None retry_config_mode=sdk_default_or_unset pool_max_connections=100 pool_max_keepalive=20 pool_keepalive_expiry=5.0s tls=trust_store=system-trust mtls_cert=none
INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_start operation_id=d9d14056-b391-4555-9a43-ff331a5e56b5 chunk_count=4 concurrency=5 allow_failed=False client_timeout_seconds=None future_timeout_seconds=3605 num_waves=1
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HT

Title
Attention Is All You Need
--------------------------------------------------
Table
Ashish Vaswani∗ Noam Shazeer∗ Niki Parmar∗ Jakob Uszkoreit∗ Google Brain Google Brain Google Research Google Research avaswani@google.com noam@google.com nikip@google.com usz@google.com Llion Jones∗ Aidan N. Gomez∗ † Łukasz Kaiser∗ Google Research University of Toronto Google Brain llion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com Illia Polosukhin∗ ‡ illia.polosukhin@gmail.com
--------------------------------------------------
Title
Abstract
--------------------------------------------------
NarrativeText
The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions en

In [3]:
print(elements[0])

{'type': 'Title', 'element_id': '2557baca4f1321e7652e9605b05323e4', 'text': 'Attention Is All You Need', 'metadata': {'is_extracted': 'true', 'filetype': 'application/pdf', 'languages': ['eng'], 'page_number': 1, 'filename': 'attention_is_all_you_need.pdf'}}


In [ ]:
#Chunk by Recursive Text Splitting

useful_types = {
    "Title",
    "NarrativeText",
    "ListItem",
    "Table"
}

filtered = [
    e for e in elements
    if e["type"] in useful_types
]

In [6]:
document_text = "\n\n".join(
    e["text"]
    for e in filtered
)

In [7]:
document_text[:1000]

'Attention Is All You Need\n\nAshish Vaswani∗ Noam Shazeer∗ Niki Parmar∗ Jakob Uszkoreit∗ Google Brain Google Brain Google Research Google Research avaswani@google.com noam@google.com nikip@google.com usz@google.com Llion Jones∗ Aidan N. Gomez∗ † Łukasz Kaiser∗ Google Research University of Toronto Google Brain llion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com Illia Polosukhin∗ ‡ illia.polosukhin@gmail.com\n\nAbstract\n\nThe dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring signiﬁcantly less time to train.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_text(document_text)

In [10]:
chunks[:5]

['Attention Is All You Need\n\nAshish Vaswani∗ Noam Shazeer∗ Niki Parmar∗ Jakob Uszkoreit∗ Google Brain Google Brain Google Research Google Research avaswani@google.com noam@google.com nikip@google.com usz@google.com Llion Jones∗ Aidan N. Gomez∗ † Łukasz Kaiser∗ Google Research University of Toronto Google Brain llion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com Illia Polosukhin∗ ‡ illia.polosukhin@gmail.com\n\nAbstract',
 'Abstract\n\nThe dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring signiﬁcantly less 

In [11]:
# Chunk By Section + Recursive Text Splitting

sections = []

current_title = None
current_text = []

for e in filtered:

    if e["type"] == "Title":

        if current_title and current_text:
            sections.append({
                "title": current_title,
                "text": "\n".join(current_text)
            })

        current_title = e["text"]
        current_text = []

    else:
        current_text.append(e["text"])

# add last section
if current_title and current_text:
    sections.append({
        "title": current_title,
        "text": "\n".join(current_text)
    })

In [12]:
sections[:5]

[{'title': 'Attention Is All You Need',
  'text': 'Ashish Vaswani∗ Noam Shazeer∗ Niki Parmar∗ Jakob Uszkoreit∗ Google Brain Google Brain Google Research Google Research avaswani@google.com noam@google.com nikip@google.com usz@google.com Llion Jones∗ Aidan N. Gomez∗ † Łukasz Kaiser∗ Google Research University of Toronto Google Brain llion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com Illia Polosukhin∗ ‡ illia.polosukhin@gmail.com'},
 {'title': 'Abstract',
  'text': 'The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and

In [13]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = []

for section in sections:

    section_chunks = splitter.split_text(
        section["text"]
    )

    for idx, chunk in enumerate(section_chunks):

        chunks.append({
            "section": section["title"],
            "chunk_id": idx,
            "text": chunk
        })

In [14]:
chunks[:5]

[{'section': 'Attention Is All You Need',
  'chunk_id': 0,
  'text': 'Ashish Vaswani∗ Noam Shazeer∗ Niki Parmar∗ Jakob Uszkoreit∗ Google Brain Google Brain Google Research Google Research avaswani@google.com noam@google.com nikip@google.com usz@google.com Llion Jones∗ Aidan N. Gomez∗ † Łukasz Kaiser∗ Google Research University of Toronto Google Brain llion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com Illia Polosukhin∗ ‡ illia.polosukhin@gmail.com'},
 {'section': 'Abstract',
  'chunk_id': 0,
  'text': 'The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quali

In [27]:
import json
from pathlib import Path

from unstructured_client import UnstructuredClient
from unstructured_client.models import operations, shared


def partition_all_pdfs(api_key: str):

    client = UnstructuredClient(
        api_key_auth=api_key
    )

    pdf_files = list(
        Path("/workspaces/nlp-paper-rag/data/papers").rglob("*.pdf")
    )

    print(f"Found {len(list(pdf_files))} PDF files to partition.")
    Path("data/partitioned").mkdir(
        parents=True,
        exist_ok=True
    )

    for pdf_path in pdf_files:

        output_file = (
            Path("data/partitioned")
            / f"{pdf_path.stem}.json"
        )

        if output_file.exists():
            print(f"Skipping: {pdf_path.name}")
            continue

        print(f"Partitioning: {pdf_path.name}")

        with open(pdf_path, "rb") as f:

            files = shared.Files(
                content=f.read(),
                file_name=pdf_path.name
            )

        req = operations.PartitionRequest(
            partition_parameters=shared.PartitionParameters(
                files=files,
                strategy="hi_res"
            )
        )

        try:

            result = client.general.partition(request=req)

            with open(output_file, "w") as f:
                json.dump(result.elements, f, indent=2)

            print(f"Saved: {output_file.name}")

        except Exception as e:

            print(f"Failed: {pdf_path.name}")
            print(e)

    print("Done.")

In [30]:
partition_all_pdfs("sA2fqF1bKetthL0H5B9zbERuc6Ffon")

INFO: split_pdf event=plan_created operation_id=ae6c5227-a96d-4293-893e-0602b75ac37b filename=sparse_dense_and_attentional_representation_for_text_retrieval.pdf strategy=hi_res page_range=1-17 page_count=17 split_size=4 chunk_count=5 concurrency=5 allow_failed=False cache_mode=disabled timeout_seconds=None retry_config_mode=sdk_default_or_unset pool_max_connections=100 pool_max_keepalive=20 pool_keepalive_expiry=5.0s tls=trust_store=system-trust mtls_cert=none


Found 25 PDF files to partition.
Skipping: roberta.pdf
Skipping: review_of_bert_based_models.pdf
Skipping: bert_review.pdf
Skipping: distil_bert.pdf
Skipping: what_the_mask.pdf
Skipping: attention_is_all_you_need.pdf
Skipping: generative_adversarial_transformers.pdf
Skipping: transformer_in_transformer.pdf
Skipping: reformer.pdf
Skipping: huggingface_transformers.pdf
Skipping: time_series_transformer.pdf
Skipping: generative_pretrained_transformer.pdf
Skipping: gptq.pdf
Skipping: graph_gpt.pdf
Skipping: bio_gpt.pdf
Skipping: graph_based_reranking.pdf
Skipping: dense_text_retrieval.pdf
Skipping: colbert.pdf
Skipping: improve_recall_for_document_retrieval_syste,s.pdf
Partitioning: sparse_dense_and_attentional_representation_for_text_retrieval.pdf


INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_start operation_id=ae6c5227-a96d-4293-893e-0602b75ac37b chunk_count=5 concurrency=5 allow_failed=False client_timeout_seconds=None future_timeout_seconds=3605 num_waves=1
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_complete operation_id=ae6c5227-a96d-4293-893e-0602b75ac37b chunk_count=5 success_count=5 failure_count=0 transport_failure_count=0 elapsed_ms=19955 allow_failed=False


Saved: sparse_dense_and_attentional_representation_for_text_retrieval.json
Partitioning: best_practices_in_Rag.pdf


INFO: split_pdf event=plan_created operation_id=eb48119e-3114-4817-8402-c04e75a7db27 filename=best_practices_in_Rag.pdf strategy=hi_res page_range=1-21 page_count=21 split_size=5 chunk_count=5 concurrency=5 allow_failed=False cache_mode=disabled timeout_seconds=None retry_config_mode=sdk_default_or_unset pool_max_connections=100 pool_max_keepalive=20 pool_keepalive_expiry=5.0s tls=trust_store=system-trust mtls_cert=none
INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_start operation_id=eb48119e-3114-4817-8402-c04e75a7db27 chunk_count=5 concurrency=5 allow_failed=False client_timeout_seconds=None future_timeout_seconds=3605 num_waves=1
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP R

...

Saved: rag_for_knowledge_intensive_tasks.json
Partitioning: corag.pdf


INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_start operation_id=b754cb67-31b6-4c26-8db9-699b6d22fbb7 chunk_count=5 concurrency=5 allow_failed=False client_timeout_seconds=None future_timeout_seconds=3605 num_waves=1
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_complete operation_id=b754cb67-31b6-4c26-8db9-699b6d22fbb7 chunk_count=5 success_count=5 failure_count=0 transport_failure_count=0 elapsed_ms=32148 allow_failed=False


Saved: corag.json
Partitioning: graph_rag.pdf


INFO: split_pdf event=plan_created operation_id=161a8d29-5dd1-4756-8860-347504069cc1 filename=graph_rag.pdf strategy=hi_res page_range=1-26 page_count=26 split_size=6 chunk_count=5 concurrency=5 allow_failed=False cache_mode=disabled timeout_seconds=None retry_config_mode=sdk_default_or_unset pool_max_connections=100 pool_max_keepalive=20 pool_keepalive_expiry=5.0s tls=trust_store=system-trust mtls_cert=none
INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_start operation_id=161a8d29-5dd1-4756-8860-347504069cc1 chunk_count=5 concurrency=5 allow_failed=False client_timeout_seconds=None future_timeout_seconds=3605 num_waves=1
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST

Saved: graph_rag.json
Done.


In [1]:
import json
from pathlib import Path

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)


def create_recursive_chunks():

    useful_types = {
        "Title",
        "NarrativeText",
        "ListItem",
        "Table"
    }

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    all_chunks = []

    partitioned_files = Path(
        "data/partitioned"
    ).glob("*.json")

    for file in partitioned_files:

        with open(file) as f:
            elements = json.load(f)

        document_text = "\n\n".join(
            e["text"]
            for e in elements
            if e["type"] in useful_types
            and e.get("text")
        )

        chunks = splitter.split_text(
            document_text
        )

        for idx, chunk in enumerate(chunks):

            all_chunks.append({
                "paper": file.stem,
                "chunk_id": idx,
                "text": chunk
            })

    Path("data/chunks").mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        "data/chunks/recursive_chunks.json",
        "w"
    ) as f:

        json.dump(
            all_chunks,
            f,
            indent=2
        )

    print(
        f"Saved {len(all_chunks)} chunks"
    )

In [2]:
create_recursive_chunks()

Saved 2441 chunks


In [3]:
def create_section_recursive_chunks():

    useful_types = {
        "Title",
        "NarrativeText",
        "ListItem",
        "Table"
    }

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    all_chunks = []

    partitioned_files = Path(
        "data/partitioned"
    ).glob("*.json")

    for file in partitioned_files:

        with open(file) as f:
            elements = json.load(f)

        filtered = [
            e for e in elements
            if e["type"] in useful_types
        ]

        sections = []

        current_title = None
        current_text = []

        for e in filtered:

            if e["type"] == "Title":

                if current_title and current_text:

                    sections.append({
                        "title": current_title,
                        "text": "\n".join(
                            current_text
                        )
                    })

                current_title = e["text"]
                current_text = []

            else:

                current_text.append(
                    e["text"]
                )

        if current_title and current_text:

            sections.append({
                "title": current_title,
                "text": "\n".join(
                    current_text
                )
            })

        chunk_id = 0

        for section in sections:

            section_chunks = splitter.split_text(
                section["text"]
            )

            for chunk in section_chunks:

                all_chunks.append({
                    "paper": file.stem,
                    "section": section["title"],
                    "chunk_id": chunk_id,
                    "text": chunk
                })

                chunk_id += 1

    Path("data/chunks").mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        "data/chunks/section_recursive_chunks.json",
        "w"
    ) as f:

        json.dump(
            all_chunks,
            f,
            indent=2
        )

    print(
        f"Saved {len(all_chunks)} chunks"
    )

In [4]:
create_section_recursive_chunks()

Saved 2525 chunks


In [6]:
import json
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

/workspaces/nlp-paper-rag/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import json
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer


def create_embeddings(
    chunk_file: str,
    model_name: str,
    output_name: str
):
    """
    Parameters
    ----------
    chunk_file:
        data/chunks/recursive_chunks.json
        OR
        data/chunks/section_recursive_chunks.json

    model_name:
        e.g. BAAI/bge-base-en-v1.5

    output_name:
        recursive
        section_recursive
    """

    print(f"Loading chunks from {chunk_file}")

    with open(chunk_file, "r") as f:
        chunks = json.load(f)

    # ------------------------------------------------------------------
    # Standardize metadata schema
    # ------------------------------------------------------------------
    for global_id, chunk in enumerate(chunks):

        # Existing fields (if missing, create them)
        chunk.setdefault("paper", None)
        chunk.setdefault("section", None)
        chunk.setdefault("chunk_id", global_id)
        chunk.setdefault("text", "")

        # Useful metadata for evaluation
        chunk["global_chunk_id"] = global_id
        chunk["strategy"] = output_name
        chunk["embedding_model"] = model_name.split("/")[-1]

    texts = [
        chunk["text"]
        for chunk in chunks
    ]

    print(f"Loading model: {model_name}")

    model = SentenceTransformer(model_name)

    print(f"Embedding {len(texts)} chunks...")

    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    model_folder = model_name.split("/")[-1]

    output_dir = (
        Path("data/embeddings")
        / model_folder
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # ---------------------------------------------------------
    # Save embeddings
    # ---------------------------------------------------------
    np.save(
        output_dir /
        f"{output_name}_embeddings.npy",
        embeddings
    )

    # ---------------------------------------------------------
    # Save metadata
    # ---------------------------------------------------------
    with open(
        output_dir /
        f"{output_name}_metadata.json",
        "w"
    ) as f:

        json.dump(
            chunks,
            f,
            indent=2
        )

    print(f"Saved embeddings to {output_dir}")

/workspaces/nlp-paper-rag/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
create_embeddings(
    chunk_file=
    "data/chunks/section_recursive_chunks.json",

    model_name=
    "BAAI/bge-large-en-v1.5",

    output_name=
    "section"
)

Loading chunks from data/chunks/section_recursive_chunks.json
Loading model: BAAI/bge-large-en-v1.5


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4974.11it/s]


Embedding 2525 chunks...


Batches: 100%|██████████| 79/79 [1:56:17<00:00, 88.33s/it]   


Saved embeddings to data/embeddings/bge-large-en-v1.5


In [2]:
create_embeddings(
    chunk_file="data/chunks/recursive_chunks.json",
    model_name="BAAI/bge-base-en-v1.5",
    output_name="recursive"
)

Loading chunks from data/chunks/recursive_chunks.json
Loading model: BAAI/bge-base-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5306.78it/s]


Embedding 2441 chunks...


Batches: 100%|██████████| 77/77 [33:56<00:00, 26.45s/it] 


Saved embeddings to data/embeddings/bge-base-en-v1.5


In [3]:
create_embeddings(
    chunk_file="data/chunks/section_recursive_chunks.json",
    model_name="BAAI/bge-base-en-v1.5",
    output_name="section"
)

Loading chunks from data/chunks/section_recursive_chunks.json
Loading model: BAAI/bge-base-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4827.23it/s]


Embedding 2525 chunks...


Batches: 100%|██████████| 79/79 [33:41<00:00, 25.59s/it]

Saved embeddings to data/embeddings/bge-base-en-v1.5


In [1]:
import json
import faiss
import numpy as np
from pathlib import Path

In [2]:
def create_faiss_index(
    embeddings_file: str,
    output_file: str
):

    embeddings = np.load(
        embeddings_file
    ).astype("float32")

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(
        dimension
    )

    index.add(embeddings)

    Path(output_file).parent.mkdir(
        parents=True,
        exist_ok=True
    )

    faiss.write_index(
        index,
        output_file
    )

    print(
        f"Saved index: {output_file}"
    )

    print(
        f"Vectors indexed: {index.ntotal}"
    )

In [4]:
create_faiss_index(
    embeddings_file="data/embeddings/bge-base-en-v1.5/recursive_embeddings.npy",
    output_file="data/faiss/bge-base-en-v1.5/recursive.index"
)

Saved index: data/faiss/bge-base-en-v1.5/recursive.index
Vectors indexed: 2441


In [5]:
create_faiss_index(
    embeddings_file="data/embeddings/bge-base-en-v1.5/section_embeddings.npy",
    output_file="data/faiss/bge-base-en-v1.5/section.index"
)

Saved index: data/faiss/bge-base-en-v1.5/section.index
Vectors indexed: 2525


In [7]:
from pathlib import Path

for p in Path("data/embeddings").rglob("*"):
    print(p)

data/embeddings/bge-base-en-v1.5
data/embeddings/bge-large-en-v1.5
data/embeddings/bge-base-en-v1.5/recursive_metadata.json
data/embeddings/bge-base-en-v1.5/recursive_embeddings.npy
data/embeddings/bge-large-en-v1.5/section_metadata.json
data/embeddings/bge-large-en-v1.5/section_embeddings.npy


In [7]:
def retrieve(
    query: str,
    model_name: str,
    index_path: str,
    metadata_path: str,
    top_k: int = 5,
):
    """
    Retrieve the top-k most similar chunks for a query.

    Parameters
    ----------
    query : str
        User query.

    model_name : str
        HuggingFace embedding model.

    index_path : str
        Path to the FAISS index.

    metadata_path : str
        Path to the metadata JSON corresponding to the embeddings.

    top_k : int
        Number of chunks to retrieve.
    """

    # Load embedding model
    model = SentenceTransformer(model_name)

    # Load FAISS index
    index = faiss.read_index(index_path)

    # Load metadata
    with open(metadata_path, "r") as f:
        metadata = json.load(f)

    # Embed query
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    # Search
    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        chunk = metadata[idx].copy()
        chunk["score"] = float(score)

        results.append(chunk)

    return results

In [8]:
results = retrieve(
    query="What is self-attention?",
    model_name="BAAI/bge-base-en-v1.5",
    index_path="data/faiss/bge-base-en-v1.5/recursive.index",
    metadata_path="data/embeddings/bge-base-en-v1.5/recursive_metadata.json",
    top_k=5
)

Loading weights: 100%|██████████| 199/199 [00:04<00:00, 41.55it/s]


In [9]:
results

[{'paper': 'attention_is_all_you_need',
  'chunk_id': 6,
  'text': 'Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 22, 23, 19].\n\nEnd-to-end memory networks are based on a recurrent attention mechanism instead of sequence- aligned recurrence and have been shown to perform well on simple-language question answering and language modeling tasks [28].\n\nTo the best of our knowledge, however, the Transformer is the ﬁrst transduction model relying entirely on self-attention to compute representations of its input and output without using sequence- aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate self-attention an

In [10]:
results = retrieve(
    query="What is self-attention?",
    model_name="BAAI/bge-base-en-v1.5",
    index_path="data/faiss/bge-base-en-v1.5/section.index",
    metadata_path="data/embeddings/bge-base-en-v1.5/section_metadata.json",
    top_k=5
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6724.89it/s]


In [11]:
results

[{'paper': 'attention_is_all_you_need',
  'section': '4 Why Self-Attention',
  'chunk_id': 25,
  'text': 'As side beneﬁt, self-attention could yield more interpretable models. We inspect attention distributions from our models and present and discuss examples in the appendix. Not only do individual attention heads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic and semantic structure of the sentences.',
  'score': 0.6766306161880493},
 {'paper': 'attention_is_all_you_need',
  'section': '2 Background',
  'chunk_id': 6,
  'text': 'Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 22, 23, 19].\nEnd-to-end memory 

In [2]:
import json
from pathlib import Path


def create_benchmark():

    question_dir = Path("/workspaces/nlp-paper-rag/data/questions")

    benchmark = []

    question_files = sorted(
        question_dir.glob("*_questions.json")
    )

    for file in question_files:

        paper_group = file.stem.replace(
            "_questions",
            ""
        )

        with open(file, "r") as f:
            questions = json.load(f)

        for q in questions:

            q["paper_group"] = paper_group

            benchmark.append(q)

    output_file = question_dir / "benchmark.json"

    with open(output_file, "w") as f:

        json.dump(
            benchmark,
            f,
            indent=2
        )

    print(f"Total Questions: {len(benchmark)}")
    print(f"Saved: {output_file}")

In [3]:
create_benchmark()

Total Questions: 470
Saved: /workspaces/nlp-paper-rag/data/questions/benchmark.json
